In [8]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/processed/features.csv')
print(df.shape)
df.head()

(337, 22)


,Year,GrandPrix,Driver,FP1_best,FP1_mean,FP1_std,FP2_best,FP2_mean,FP2_std,FP3_best,...,track_evolution,AirTemp,TrackTemp,Humidity,Rainfall,Team,quali_best,Driver_enc,Team_enc,GrandPrix_enc
0,2023,Australian Grand Prix,ALB,79.766,80.581667,0.982823,81.182,81.182,NaN,78.553,...,1.213,15.204211,22.712632,53.4,1,Williams,77.609,0,9,0
1,2023,Australian Grand Prix,ALO,79.317,82.208889,1.773777,78.887,79.657,1.088944,77.727,...,1.590,15.204211,22.712632,53.4,1,Aston Martin,77.139,1,3,0
2,2023,Australian Grand Prix,BOT,80.419,81.691000,0.852589,80.312,81.489,1.496578,78.809,...,1.610,15.204211,22.712632,53.4,1,Alfa Romeo,78.714,2,0,0
3,2023,Australian Grand Prix,DEV,79.933,81.472000,1.070387,80.600,80.896,0.503191,79.092,...,0.841,15.204211,22.712632,53.4,1,AlphaTauri,78.335,3,1,0
4,2023,Australian Grand Prix,GAS,79.646,81.398200,1.718447,80.206,80.944,0.851820,78.094,...,1.552,15.204211,22.712632,53.4,1,Alpine,77.574,4,2,0


In [9]:
FEATURES = [
    'FP1_best', 'FP2_best', 'FP3_best',
    'FP1_mean', 'FP2_mean', 'FP3_mean',
    'FP1_std', 'FP2_std', 'FP3_std',
    'track_evolution',
    'AirTemp', 'TrackTemp', 'Humidity', 'Rainfall',
    'Driver_enc', 'Team_enc', 'GrandPrix_enc'
]
TARGET = 'quali_best'

# Fill any remaining NaN values
df[FEATURES] = df[FEATURES].fillna(df[FEATURES].median())

# Since we only have 2023 data, we split 80/20 by race order instead
split_idx = int(len(df) * 0.8)
train = df.iloc[:split_idx]
test  = df.iloc[split_idx:]

X_train, y_train = train[FEATURES], train[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]

print(f"Train size: {len(train)} | Test size: {len(test)}")

Train size: 269 | Test size: 68


In [10]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score

models = {
    'Ridge': Ridge(),
    'RF':    RandomForestRegressor(n_estimators=200, random_state=42),
    'GBM':   GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, random_state=42),
}

trained_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    r2  = r2_score(y_test, preds)
    trained_models[name] = model
    print(f"{name:10s} | MAE: {mae:.3f}s | R²: {r2:.4f}")

Ridge      | MAE: 7.683s | R²: -0.0295
RF         | MAE: 3.709s | R²: 0.6437
GBM        | MAE: 3.953s | R²: 0.6356


In [13]:
import joblib
import json
import os

# Use the exact absolute path
models_dir = '/Users/harshiniratnakumar/Desktop/f1-quali-predictor/models'
os.makedirs(models_dir, exist_ok=True)

print("Saving to:", models_dir)

# Save model
joblib.dump(best_model, os.path.join(models_dir, 'best_model.pkl'))

# Save feature columns
FEATURE_COLS = [
    'FP1_best', 'FP2_best', 'FP3_best',
    'FP1_mean', 'FP2_mean', 'FP3_mean',
    'FP1_std',  'FP2_std',  'FP3_std',
    'track_evolution',
    'AirTemp', 'TrackTemp', 'Humidity', 'Rainfall',
    'Driver_enc', 'Team_enc', 'GrandPrix_enc'
]
with open(os.path.join(models_dir, 'feature_cols.json'), 'w') as f:
    json.dump(FEATURE_COLS, f)

print("✅ Model saved!")
print("✅ Feature columns saved!")

Saving to: /Users/harshiniratnakumar/Desktop/f1-quali-predictor/models
✅ Model saved!
✅ Feature columns saved!
